In [9]:
import importlib
from pathlib import Path
import numpy as np
import pandas as pd
import project_settings
importlib.reload(project_settings)
from project_settings import PATH_WRK, BASE, YEARS, RESULTS_DIR, SRC_DIR, SRC_DIRS, CHOC_SCENARIO

CONTENT = "dir"        # "dir" : rd_dir_CI | "tot" : rd_tot_CI
YEAR = 2020 # YEARS[0]       # year of the base table (the same for all the years of YEARS_CALC)
YEARS_CALC = list(range(2025, 2051))                   # years computed: the shock of year t is the level gap vs the baseline of year t (nothing to cumulate)
DISPLAY_YEARS = [2025, 2030, 2035, 2040, 2045, 2050]   # years displayed in the table (among YEARS_CALC); YEAR = base year, variation 0
OMEGA_FILE = "omega_progressif.csv"                               # inputs/: origin, dest, sector, delta_pct, + year column for one ω per year (without a year column: the same ω every year)

EU27 = ["AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR", "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT",
        "RO", "SK", "SI", "ES", "SE"]
BLOCKS = {
    "USA": ["US"],
    "EU27": EU27,
    "RestOECD": ["AU", "CA", "JP", "KR", "MX", "NZ", "NO", "CH", "UK"],
    "China": ["CN"],
    "RestBRICS": ["BR", "RU", "IN", "ZA"],
    "RestAsia": ["BD", "KH", "ID", "LA", "MY", "MM", "NP", "PK", "PH", "SG", "TW", "TJ", "TL", "TH", "VN"],
}
REGION_TO_BLOCK = {region: block for block, regions in BLOCKS.items() for region in regions}
DEFAULT_BLOCK = "RoW"

MANUFACTURING = ["C13_C15", "C16_C18", "C20+C21", "C22", "C23", "C24", "C25", "C26", "C27", "C28", "C29", "C30", "C31_C33"]
NONMANUF = "NONMANUF"
AGGREGATES = ["C", "H"]    # sector groups (prefix of the sector codes) also computed as one aggregate sector, as add_sector() of the 2.0

# ---- final file in IAMC format: Model | Scenario | Region | Variable | Unit | one column per year -----------------------------------
# Combined export: Region is the importer; Variable is the sector total or "sector | supplying bloc".
IAMC_MODEL = "NEMESIS-World v1.1"                                     # "Model" column
IAMC_SCENARIO = "model{model}_case{scenario}"  # "Scenario" column: {model} = 2 / 3 / 4 (model of blocks), {scenario} = A / B
IAMC_VARIABLE = "{sector} | {origin}"                        # "Variable" column: {sector} = sector whose unit cost changes ("C" = all manufacturing), {origin} = bloc supplying the inputs
# English NACE Rev. 2 labels and project aggregates, applied only to the final export.
# H includes H53+J in the current project aggregation.
SECTOR_NAMES = {'A': 'Agriculture, forestry and fishing',
 'B': 'Mining and quarrying',
 'C': 'Manufacturing',
 'C10_C12': 'Food, beverages and tobacco',
 'C13_C15': 'Textiles, wearing apparel and leather products',
 'C16_C18': 'Wood, paper and printing products',
 'C19': 'Coke and refined petroleum products',
 'C20+C21': 'Chemicals and pharmaceuticals',
 'C22': 'Rubber and plastic products',
 'C23': 'Other non-metallic mineral products',
 'C24': 'Basic metals',
 'C25': 'Fabricated metal products',
 'C26': 'Computer, electronic and optical products',
 'C27': 'Electrical equipment',
 'C28': 'Machinery and equipment n.e.c.',
 'C29': 'Motor vehicles, trailers and semi-trailers',
 'C30': 'Other transport equipment',
 'C31_C33': 'Furniture, other manufacturing, machinery repair and installation',
 'D': 'Electricity, gas, steam and air conditioning supply',
 'E': 'Water supply, sewerage, waste management and remediation',
 'F': 'Construction',
 'G': 'Wholesale and retail trade, motor vehicle repair',
 'H': 'Transportation, storage, information and communication services',
 'H49': 'Land transport and transport via pipelines',
 'H50': 'Water transport',
 'H51': 'Air transport',
 'H52': 'Warehousing and support activities for transportation',
 'H53+J': 'Postal, courier, information and communication services',
 'I': 'Accommodation and food service activities',
 'K': 'Financial and insurance activities',
 'L': 'Real estate activities',
 'M+N': 'Professional, scientific, technical, administrative and support services',
 'O+U': 'Public administration, defence, social security and extraterritorial organisations',
 'P': 'Education',
 'Q': 'Human health and social work activities',
 'R+S+T': 'Arts, entertainment, recreation, other services and household activities'}

IAMC_YEARS = YEARS_CALC                                       # years written, one column each (among YEARS_CALC)
IAMC_CSV = dict(sep=";", decimal=",")                         # csv format (";" and "," = French Excel opens it directly)
IAMC_XLSX = True                                              # also write the same table as .xlsx

In [10]:
def read_matrix(path):
    return pd.read_csv(path, sep=";", header=[0, 1, 2], index_col=[0, 1, 2])


def flat(m):
    m = m.copy()
    m.index, m.columns = m.index.droplevel(1), m.columns.droplevel(1)
    m.index.names = m.columns.names = ["Region", "Sector"]
    return m


MODEL_BLOCKS = {
    2: {"D": (None, True), "F": (None, False)},
    3: {"I": (True, None), "X": (False, None)},
    4: {"same_same": (True, True), "same_diff": (True, False), "diff_same": (False, True), "diff_diff": (False, False)},
}


def block_matrices(m, region_to_block=REGION_TO_BLOCK, default_block=DEFAULT_BLOCK):
    reg_i, reg_j = m.index.get_level_values("Region"), m.columns.get_level_values("Region")
    foreign = np.asarray(reg_i)[:, None] != np.asarray(reg_j)[None, :]
    rows = pd.MultiIndex.from_arrays([[region_to_block.get(r, default_block) for r in reg_i], m.index.get_level_values("Sector")], names=["Bloc", "Sector"])
    cols = pd.MultiIndex.from_arrays([[region_to_block.get(r, default_block) for r in reg_j], m.columns.get_level_values("Sector")], names=["Bloc", "Sector"])
    out = []
    for mask in (~foreign, foreign):
        a = pd.DataFrame(m.to_numpy() * mask, index=rows, columns=cols)
        out.append(a.groupby(level=["Bloc", "Sector"]).sum().T.groupby(level=["Bloc", "Sector"]).sum().T)
    return out


def omega_blocks(index, columns, omega, manufacturing=MANUFACTURING):
    b_i, sec_i = np.asarray(index.get_level_values("Bloc")), np.asarray(index.get_level_values("Sector"))
    b_j = np.asarray(columns.get_level_values("Bloc"))
    om = np.ones((len(index), len(columns)))
    for r in omega.sort_values("sector", key=lambda s: s != "ALL").itertuples():
        rows = (b_i == r.origin) & ((sec_i == r.sector) | (r.sector == "ALL"))
        om[np.ix_(rows, b_j == r.dest)] = 1 + r.delta_pct / 100
    return np.where(np.isin(sec_i, manufacturing)[:, None], om, 1.0)


def shock_bilateral(cd, cf, zd, zf, omega, beta, model, aggregates=AGGREGATES, manufacturing=MANUFACTURING):
    """Same cost shock as shock_blocks + add_sector, but BY BLOC OF ORIGIN of the inputs: dlnCost of the sector x of the bloc y due to what it buys
    from the bloc z. One row per (scenario, bloc y, sector x, origin z).
    - The change of the stock S_k of a buyer is a sum over the supplying blocs z: dS_k(z) = sum over the suppliers i of the bloc z of
      c_ij * (omega_ij - 1) (same masks k as shock_blocks). Scenario B adds the substitution gain g (domestic production fills the gap), attributed
      to the origin z = bloc of the buyer itself.
    - dlnS_k = log((S0 + sum_z dS_k(z)) / S0) is not additive: it is split between the origins in proportion to their share of the stock change,
      dlnS_k(z) = dlnS_k * dS_k(z) / sum_z dS_k(z), so that the sum over z gives back the dlnS_k of shock_blocks exactly.
    - dlnCost(z) = sum_k beta_k * dlnS_k(z): the sum over z is the dlnCost of shock_blocks + add_sector.
    - aggregates (e.g. "C", "H"): the stocks and the changes of the sectors of the group are summed first (as add_sector), then dlnS and the split."""
    idx, cols = cd.index, cd.columns
    bloc_i, sec_i = np.asarray(idx.get_level_values("Bloc")), np.asarray(idx.get_level_values("Sector"))
    bloc_j, sec_j = np.asarray(cols.get_level_values("Bloc")), np.asarray(cols.get_level_values("Sector"))
    origins = np.array(list(dict.fromkeys(bloc_i)))
    n_o, n_d = len(origins), len(cols)
    members = (origins[:, None] == bloc_i[None, :]).astype(float)        # (origin, supplier): the supplier belongs to the bloc
    own = origins[:, None] == bloc_j[None, :]                              # (origin, buyer): the origin is the bloc of the buyer
    same_sec = sec_i[:, None] == sec_j[None, :]
    om = omega_blocks(idx, cols, omega, manufacturing)
    cdv, cfv = cd.to_numpy(), cf.to_numpy()
    types = MODEL_BLOCKS[model]

    s0, shock = {}, {}
    for k, (sec, cty) in types.items():
        ms = np.ones_like(same_sec) if sec is None else same_sec == sec
        d = (cdv * ms).sum(0) if cty is not False else 0.0                  # domestic part (never shocked)
        f0 = (cfv * ms).sum(0) if cty is not True else 0.0
        s0[k] = d + f0
        shock[k] = members @ (cfv * (om - 1.0) * ms) if cty is not True else np.zeros((n_o, n_d))   # change of S_k by origin bloc

    none = np.zeros(n_d)
    cases = [("A", none, none)]
    if zd is not None:                 # scenario B: needs the volumes z
        by_sector = lambda a: pd.DataFrame(a, index=idx, columns=cols).groupby(level="Sector").sum()
        c_dom, z_dom = by_sector(cdv), by_sector(zd.to_numpy())
        lost = by_sector((1 - om) * zf.to_numpy())
        rho_d = (c_dom / z_dom.where(z_dom > 0)).fillna(0.0)
        gain = np.maximum(rho_d * lost, -c_dom)
        g_same = gain.to_numpy()[gain.index.get_indexer(sec_j), np.arange(len(sec_j))]
        cases.append(("B", g_same, gain.sum().to_numpy() - g_same))

    def split(S0, dS):
        """S0: stock before; dS: change of the stock by origin (array over the origins). Returns dlnS by origin."""
        tot = dS.sum(0)
        with np.errstate(all="ignore"):
            dln = np.where(S0 > 0, np.log((S0 + tot) / S0), 0.0)
            share = np.where(tot != 0, dS / np.where(tot != 0, tot, 1.0), 0.0)
        return dln * share

    out = []
    for scenario, g_same, g_diff in cases:
        dS = {}
        for k, (sec, cty) in types.items():
            g = 0.0 if cty is False else (g_same + g_diff if sec is None else g_same if sec else g_diff)
            dS[k] = shock[k] + own * g                                       # the gain g is attributed to the bloc of the buyer
        res = pd.DataFrame({"bloc": np.tile(bloc_j, n_o), "sector": np.tile(sec_j, n_o), "origin": np.repeat(origins, n_d)})
        for k in types:
            res[f"dlnS_{k}"] = split(s0[k], dS[k]).ravel()
        parts = [res]
        for a in aggregates:
            in_a = np.array([s.startswith(a) for s in sec_j])
            for b in dict.fromkeys(bloc_j):
                sel = in_a & (bloc_j == b)
                if not sel.any():
                    continue
                row = pd.DataFrame({"bloc": b, "sector": a, "origin": origins})
                for k in types:
                    S0 = s0[k][sel].sum()
                    dS_a = (dS[k][:, sel] * (s0[k][sel] > 0)).sum(1)          # stocks without a baseline stay at 0, as add_sector
                    row[f"dlnS_{k}"] = split(S0, dS_a)
                parts.append(row)
        out.append(pd.concat(parts, ignore_index=True).assign(scenario=scenario))
    res = pd.concat(out, ignore_index=True)
    if beta is not None:
        res = res.merge(beta, on="sector", how="left")
        res["dlnCost"] = sum(res[f"beta_{k}"] * res[f"dlnS_{k}"] for k in types)
    return res

In [11]:
if BASE not in ("EXIOBASE", "FIGARO", "OECD_ICIO", "GTAP"):
    raise ValueError(f"BASE={BASE!r} : le choc utilise les flux Z d'une base MRIO (EXIOBASE, FIGARO, OECD_ICIO ou GTAP)")


Z_REF_YEAR = 2010   # year whose z_CI is used to rebuild the volumes of the years without z_CI (fixed technical coefficients)


def mpath(name, year, suffix="_agg"):
    return Path(SRC_DIR) / BASE / f"output_loop_{BASE}_{year}" / f"{name}_{BASE}_{year}{suffix}.csv"


prod = pd.read_csv(Path(SRC_DIR) / BASE / f"output_loop_{BASE}" / f"production_{BASE}_agg.csv", sep=";").set_index(["Region", "Sector"])
_cache = {}


def load_z(c, year):
    if mpath("z_CI", year).exists():
        return flat(read_matrix(mpath("z_CI", year))).reindex(index=c.index, columns=c.columns).fillna(0.0), "z_CI exact"
    if "A" not in _cache:
        for d in (SRC_DIR, PATH_WRK + "SRDV (c,s)/", PATH_WRK + "SRDQ (c,s)/"):   # z_CI of the reference year, in any run folder
            p = Path(d) / BASE / f"output_loop_{BASE}_{Z_REF_YEAR}" / f"z_CI_{BASE}_{Z_REF_YEAR}_agg.csv"
            if p.exists():
                z_ref = flat(read_matrix(p)).reindex(index=c.index, columns=c.columns).fillna(0.0)
                x_ref = prod[str(Z_REF_YEAR)].reindex(c.index).fillna(0.0).to_numpy()
                _cache["A"] = z_ref.to_numpy() / np.where(x_ref > 0, x_ref, 1.0)[None, :]
                break
        else:
            raise FileNotFoundError(f"z_CI_{BASE}_{Z_REF_YEAR}_agg.csv introuvable : lancer know_calc_loop4.0 pour l'année {Z_REF_YEAR}")
    x = prod[str(year)].reindex(c.index).fillna(0.0).to_numpy()
    z_prox = _cache["A"] * x[None, :]                                    # volumes with the technical coefficients of Z_REF_YEAR
    zs, cs = z_prox.sum(axis=1), c.to_numpy().sum(axis=1)
    rho = np.divide(cs, zs, out=np.zeros(len(zs)), where=zs > 0)         # R&D of the supplier per euro sold this year
    z = np.where(rho[:, None] > 0, np.divide(c.to_numpy(), rho[:, None], out=np.zeros(c.shape), where=rho[:, None] > 0), z_prox)   # z = c / rho
    return pd.DataFrame(z, index=c.index, columns=c.columns), f"z reconstruit (c/rho, A {Z_REF_YEAR})"


def find(name):
    for d in SRC_DIRS:
        p = Path(d) / BASE / f"output_loop_{BASE}_{YEAR}" / f"{name}_{BASE}_{YEAR}_agg.csv"
        if p.exists():
            return p
    raise FileNotFoundError(f"{name}_{BASE}_{YEAR}_agg.csv introuvable dans {SRC_DIRS} : lancer know_calc_loop puis aggregate_to_common_mapping")


inputs = Path(PATH_WRK) / "choc_tfp" / "inputs"
omega_all = pd.read_csv(inputs / OMEGA_FILE)                                      # ω of all the years (year column optional)

prefix = {"dir": "rd_dir_CI", "tot": "rd_tot_CI"}[CONTENT]
c0 = flat(read_matrix(find(prefix)))
if CHOC_SCENARIO == "B":                      # only scenario B needs the volumes z (A does not)
    z0, z_src = load_z(c0, YEAR)
else:
    z0, z_src = None, "scénario A : pas de volumes z"
print(f"volumes {YEAR} : {z_src}")

cd, cf = block_matrices(c0)                       # domestic / foreign content, aggregated by (block, sector) BEFORE the shock
zd, zf = (None, None) if z0 is None else [m.reindex(index=cd.index, columns=cd.columns).fillna(0.0) for m in block_matrices(z0)]
print(f"{cd.shape[0]} (bloc, secteur) fournisseurs x {cd.shape[1]} (bloc, secteur) acheteurs")

out = Path(RESULTS_DIR) / "choc_tfp"
out.mkdir(parents=True, exist_ok=True)


volumes 2020 : z_CI exact
238 (bloc, secteur) fournisseurs x 238 (bloc, secteur) acheteurs


In [12]:
def omega_of(year):
    if "year" not in omega_all.columns:
        return omega_all
    om = omega_all[omega_all["year"] == year].drop(columns="year")
    if om.empty:
        raise ValueError(f"{OMEGA_FILE} : aucune ligne pour l'année {year}")
    return om


results = {}
for model in MODEL_BLOCKS:
    f_beta = inputs / ("beta.csv" if model == 2 else f"beta_modele{model}.csv")
    beta_m = pd.read_csv(f_beta) if f_beta.exists() else None
    cache, frames = {}, []
    for year in YEARS_CALC:
        omega = omega_of(year)
        key = omega.to_csv(index=False)
        if key not in cache:
            cache[key] = shock_bilateral(cd, cf, zd, zf, omega, beta_m, model)
        frames.append(cache[key].assign(year=year))
    results[model] = pd.concat(frames, ignore_index=True)
    results[model].to_csv(out / f"resultats{model}_blocs_bilateral_{BASE}_{CONTENT}_horizon_choc{CHOC_SCENARIO}.csv", index=False)
    print(f"modèle ({model}) : {len(YEARS_CALC)} années, {len(results[model])} lignes (bloc, secteur, bloc d'origine) écrites dans {out}")

modèle (2) : 26 années, 91728 lignes (bloc, secteur, bloc d'origine) écrites dans D:\ENTICE\runs\SRDQ (c,s)\choc_tfp
modèle (3) : 26 années, 91728 lignes (bloc, secteur, bloc d'origine) écrites dans D:\ENTICE\runs\SRDQ (c,s)\choc_tfp
modèle (4) : 26 années, 91728 lignes (bloc, secteur, bloc d'origine) écrites dans D:\ENTICE\runs\SRDQ (c,s)\choc_tfp


In [13]:
# Table of the displayed years: one row per (scenario, sector, bloc, origin bloc), one column per year. Change in unit cost of the sector of the bloc
# due to what it buys from the origin bloc, relative to the baseline (the base year does not move: 0)
tables = {}
for model in MODEL_BLOCKS:
    if "dlnCost" not in results[model].columns:
        continue
    wide = results[model].pivot_table(index=["scenario", "sector", "bloc", "origin"], columns="year", values="dlnCost").reindex(columns=DISPLAY_YEARS)
    wide.insert(0, YEAR, 0.0)
    wide = (np.exp(wide) - 1) * 100
    wide.columns.name = None
    tables[model] = wide.round(4).reset_index()
    path = out / f"variation_cout_bilateral_{BASE}_{CONTENT}_modele{model}_base{YEAR}.csv"
    tables[model].to_csv(path, index=False)
    print(f"modèle ({model}) : tableau de {len(tables[model])} lignes écrit dans {path}")

tables[2]

modèle (2) : tableau de 3528 lignes écrit dans D:\ENTICE\runs\SRDQ (c,s)\choc_tfp\variation_cout_bilateral_GTAP_dir_modele2_base2020.csv
modèle (3) : tableau de 3528 lignes écrit dans D:\ENTICE\runs\SRDQ (c,s)\choc_tfp\variation_cout_bilateral_GTAP_dir_modele3_base2020.csv
modèle (4) : tableau de 3528 lignes écrit dans D:\ENTICE\runs\SRDQ (c,s)\choc_tfp\variation_cout_bilateral_GTAP_dir_modele4_base2020.csv


,scenario,sector,bloc,origin,2020,2025,2030,2035,2040,2045,2050
0,A,A,China,China,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,A,A,China,EU27,0.0,0.0214,0.1295,0.2398,0.3522,0.4669,0.5840
2,A,A,China,RestAsia,0.0,0.0013,0.0076,0.0141,0.0207,0.0274,0.0343
3,A,A,China,RestBRICS,0.0,0.0008,0.0048,0.0089,0.0130,0.0172,0.0215
4,A,A,China,RestOECD,0.0,0.0474,0.2873,0.5322,0.7823,1.0378,1.2989
...,...,...,...,...,...,...,...,...,...,...,...
3523,B,R+S+T,USA,RestAsia,0.0,0.0001,0.0004,0.0008,0.0011,0.0015,0.0019
3524,B,R+S+T,USA,RestBRICS,0.0,0.0002,0.0013,0.0024,0.0034,0.0046,0.0057
3525,B,R+S+T,USA,RestOECD,0.0,0.0081,0.0491,0.0906,0.1328,0.1755,0.2189
3526,B,R+S+T,USA,RoW,0.0,0.0003,0.0019,0.0036,0.0052,0.0069,0.0086


In [14]:
# One final table: sector totals and the detail by supplying bloc.
# Both use the SAME bilateral dlnCost values, computed in the preceding cells.
# Sector total (%) = 100 * (exp(sum over origins of dlnCost) - 1).
# Bilateral row (%) = 100 * (exp(dlnCost for this origin) - 1), as in the earlier export.
# The bilateral rows add up in LOGS, not in percentages. Do not sum the exported percentages.
# Each year is a level difference from its own baseline; nothing is cumulated over years.

IAMC_UNIT = "%"


def to_iamc(results, years=DISPLAY_YEARS):
    """Export sector totals and origin rows from a single common log-cost calculation.

    Variable = sector: total cost variation in percent.
    Variable = sector | origin: exponentiated log contribution of the supplying bloc.
    The exact reconstruction is product(1 + bilateral_percent / 100) - 1.
    These bilateral percentages retain their earlier definition and are not additive.
    """
    frames = []
    for model, res in results.items():
        if "dlnCost" not in res.columns:
            print(f"modèle ({model}) : pas de dlnCost (fichier beta absent), ignoré")
            continue
        long = res.loc[
            res["year"].isin(years),
            ["scenario", "bloc", "sector", "origin", "year", "dlnCost"],
        ].dropna(subset=["dlnCost"]).copy()

        # Sum by origin BEFORE converting log differences into percentage changes.
        total = long.groupby(
            ["scenario", "bloc", "sector", "year"], as_index=False
        )["dlnCost"].sum(min_count=1)
        total["Variable"] = [SECTOR_NAMES.get(s, s) for s in total["sector"]]

        long["Variable"] = [
            IAMC_VARIABLE.format(sector=SECTOR_NAMES.get(s, s), origin=o)
            for s, o in zip(long["sector"], long["origin"])
        ]
        for table in (total, long):
            table["value"] = (np.exp(table["dlnCost"]) - 1) * 100
            table["Case"] = [
                IAMC_SCENARIO.format(model=model, scenario=s) for s in table["scenario"]
            ]
            table["Region (importer)"] = table["bloc"]
            frames.append(table[["Case", "Region (importer)", "Variable", "year", "value"]])

    long = pd.concat(frames, ignore_index=True)
    wide = long.pivot(
        index=["Case", "Region (importer)", "Variable"], columns="year", values="value"
    ).reset_index()
    wide.columns.name = None
    wide.insert(0, "Model", IAMC_MODEL)
    wide.insert(4, "Unit", IAMC_UNIT)
    return wide


iamc = to_iamc(results)
stem = f"ENTICE_T34_{BASE}-{CONTENT}_AllCases"
iamc.to_csv(out / f"{stem}.csv", index=False, encoding="utf-8-sig", **IAMC_CSV)
is_bilateral = iamc["Variable"].str.contains("|", regex=False)
print(
    f"{len(iamc)} lignes : {(~is_bilateral).sum()} totaux sectoriels et "
    f"{is_bilateral.sum()} lignes bilatérales écrites dans {out / stem}.csv"
)
iamc.head(10)


12096 lignes : 1512 totaux sectoriels et 10584 lignes bilatérales écrites dans D:\ENTICE\runs\SRDQ (c,s)\choc_tfp\ENTICE_T34_GTAP-dir_AllCases.csv


,Model,Case,Region (importer),Variable,Unit,2025,2030,2035,2040,2045,2050
0,NEMESIS-World v1.1,model2_caseA,China,Accommodation and food service activities,%,0.022822,0.137677,0.253780,0.371151,0.489829,0.609836
1,NEMESIS-World v1.1,model2_caseA,China,Accommodation and food service activities | China,%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,NEMESIS-World v1.1,model2_caseA,China,Accommodation and food service activities | EU27,%,0.004873,0.029386,0.054142,0.079146,0.104406,0.129924
3,NEMESIS-World v1.1,model2_caseA,China,Accommodation and food service activities | Re...,%,0.000262,0.001578,0.002908,0.004250,0.005606,0.006975
4,NEMESIS-World v1.1,model2_caseA,China,Accommodation and food service activities | Re...,%,0.000162,0.000976,0.001799,0.002629,0.003468,0.004315
5,NEMESIS-World v1.1,model2_caseA,China,Accommodation and food service activities | Re...,%,0.013658,0.082374,0.151805,0.221962,0.292866,0.364530
6,NEMESIS-World v1.1,model2_caseA,China,Accommodation and food service activities | RoW,%,0.000237,0.001426,0.002628,0.003841,0.005066,0.006303
7,NEMESIS-World v1.1,model2_caseA,China,Accommodation and food service activities | USA,%,0.003629,0.021881,0.040314,0.058931,0.077735,0.096732
8,NEMESIS-World v1.1,model2_caseA,China,"Agriculture, forestry and fishing",%,0.089468,0.542902,1.006766,1.481476,1.967546,2.465450
9,NEMESIS-World v1.1,model2_caseA,China,"Agriculture, forestry and fishing | China",%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [15]:
# ---- VA weighting of the exported cost variations ------------------------------------------------------------------------------------
# Reads the CSV written by the cell above (no new file is written): the aggregates of the CSV are not weighted by the value added. Here the sectors are
# averaged with the STAN VA of the sector IN THE BLOC (VA of the STAN countries of the bloc, summed) as weight, for the manufacturing and for all the sectors.
# The average is made on the log cost variations (dlnCost = log(1 + % / 100), exact: the CSV keeps the full precision), then converted back into %.
# Where there is no VA, there is no weighted value (NaN): blocs without any STAN country (China, RestBRICS, RestAsia, RoW); inside a bloc, sectors without VA
# are left out and the weights are renormalised on the sectors that have one.
VA_FILE = next(p for p in Path(PATH_WRK, "code").iterdir() if p.name.startswith("cr") and "panel" in p.name) / "Panels_intermediaires" / "panel_va_STAN.csv"
VA_VAR = "VA_V"                                               # VA_V = current euros | VA_Q = volume
VA_YEAR = YEAR                                                # year of the weights; for a (country, sector) without VA this year, the latest earlier year is used
VA_GROUPS = {                                                 # aggregate: sectors averaged (the aggregates "C" and "H" of AGGREGATES are not components)
    "Manufacturing": lambda s: s.startswith("C") and s not in AGGREGATES,
    "All sectors": lambda s: s not in AGGREGATES,
}
SEP = " | "                                                   # separator of "sector | origin" in the Variable column (IAMC_VARIABLE)


def load_va_weights(path=VA_FILE, var=VA_VAR, year=VA_YEAR, region_to_block=REGION_TO_BLOCK, default_block=DEFAULT_BLOCK):
    va = pd.read_csv(path, usecols=["Region", "Sector", "annee", var])
    va = va[(va["annee"] <= year) & (va[var] > 0)].dropna(subset=[var])
    last = va.sort_values("annee").groupby(["Region", "Sector"], as_index=False).last()      # latest year <= year, per (country, sector)
    last["bloc"] = last["Region"].map(region_to_block).fillna(default_block)
    weights = last.groupby(["bloc", "Sector"])[var].sum().rename("va").reset_index().rename(columns={"Sector": "sector"})
    return weights, last.groupby("Region")["annee"].agg(["min", "max"])


def average(long, groups, weights=None):
    """Average of the log variations of the sectors of each group, by (case, bloc, origin, year): weights = VA of the sector in the bloc, or simple average
    if weights is None. The result is converted back into %."""
    base = long.dropna(subset=["dln"])
    base = base.assign(va=1.0) if weights is None else base.merge(weights, on=["bloc", "sector"], how="inner")
    keys, res = ["Case", "bloc", "origin", "year"], []
    for label, in_group in groups.items():
        sub = base[base["sector"].map(in_group)].assign(wx=lambda d: d["va"] * d["dln"])
        agg = sub.groupby(keys).agg(wx=("wx", "sum"), w=("va", "sum")).reset_index()
        agg["value"], agg["Group"] = (np.exp(agg["wx"] / agg["w"]) - 1) * 100, label
        res.append(agg[keys + ["Group", "value"]])
    return pd.concat(res, ignore_index=True)


# the CSV of the cell above, back in long format: one row per (case, bloc, sector or "C" aggregate, origin or "Total", year)
iamc_csv = pd.read_csv(out / f"{stem}.csv", encoding="utf-8-sig", **IAMC_CSV)
name_to_code = {name: code for code, name in SECTOR_NAMES.items()}
long = iamc_csv.melt(id_vars=["Case", "Region (importer)", "Variable"], value_vars=[c for c in iamc_csv.columns if str(c).isdigit()],
                     var_name="year", value_name="value").rename(columns={"Region (importer)": "bloc"})
part = long["Variable"].str.partition(SEP)
long["sector"] = part[0].map(name_to_code).fillna(part[0])
long["origin"] = part[2].replace("", "Total")                  # no origin = total of the sector
long["year"] = long["year"].astype(int)
long["dln"] = np.log1p(long["value"] / 100)

va_weights, va_years = load_va_weights()
no_va = sorted(set(long["bloc"]) - set(va_weights["bloc"]))
print(f"VA {VA_VAR}: weights of {va_weights['bloc'].nunique()} blocs ({', '.join(sorted(va_weights['bloc'].unique()))}), "
      f"{va_years.shape[0]} STAN countries, years used {int(va_years['min'].min())}-{int(va_years['max'].max())}")
print(f"blocs without VA (no VA-weighted value): {no_va}")
sectors = long.loc[long["origin"] == "Total", "sector"].unique()
for label, in_group in VA_GROUPS.items():
    in_g = [s for s in sectors if in_group(s)]
    for b in sorted(va_weights["bloc"].unique()):
        miss = [s for s in in_g if s not in set(va_weights.loc[va_weights["bloc"] == b, "sector"])]
        print(f"{label} - {b}: {len(in_g) - len(miss)}/{len(in_g)} sectors with VA" + (f" (without: {', '.join(miss)})" if miss else ""))

# WITHOUT VA: manufacturing = aggregate "C" of the CSV (computed from the R&D stocks); all sectors = simple average of the sectors (the CSV has no such aggregate)
without_va = pd.concat([long.loc[long["sector"] == "C", ["Case", "bloc", "origin", "year", "value"]].assign(Group="Manufacturing"),
                        average(long, {"All sectors": VA_GROUPS["All sectors"]})], ignore_index=True).rename(columns={"value": "sans VA"})
with_va = average(long, VA_GROUPS, va_weights).rename(columns={"value": "avec VA"})
comparison = without_va.merge(with_va, on=["Case", "bloc", "origin", "year", "Group"], how="left")
comparison["écart"] = comparison["avec VA"] - comparison["sans VA"]          # in points of % (VA-weighted minus without VA)

VA VA_V: weights of 3 blocs (EU27, RestOECD, USA), 23 STAN countries, years used 2018-2020
blocs without VA (no VA-weighted value): ['China', 'RestAsia', 'RestBRICS', 'RoW']
Manufacturing - EU27: 15/15 sectors with VA
Manufacturing - RestOECD: 15/15 sectors with VA
Manufacturing - USA: 13/15 sectors with VA (without: C20+C21, C13_C15)
All sectors - EU27: 34/34 sectors with VA
All sectors - RestOECD: 30/34 sectors with VA (without: H51, H49, H52, H50)
All sectors - USA: 25/34 sectors with VA (without: A, C20+C21, F, D, K, M+N, L, C13_C15, E)


In [ ]:
# Comparison without / with VA weighting (in %, "écart" = with - without, in points of %). Nothing is written: tables only.
# Without VA: manufacturing = aggregate "C" of the CSV (R&D stocks); all sectors = simple average of the sectors. NaN = no VA for this bloc.
COMPARE_YEARS = [2030, 2050]                                  # years displayed (among DISPLAY_YEARS)
DETAIL_CASE = f"model4_case{CHOC_SCENARIO}"                   # case displayed with the detail by supplying bloc


def show(df, years=COMPARE_YEARS):
    wide = df[df["year"].isin(years)].pivot(index=["Case", "bloc", "Group", "origin"], columns="year", values=["sans VA", "avec VA", "écart"])
    wide = wide.swaplevel(axis=1).reindex(columns=pd.MultiIndex.from_product([years, ["sans VA", "avec VA", "écart"]])).sort_index()
    wide.index = wide.index.set_names(["Case", "Region (importer)", "Group", "Origin"])
    return wide.round(4)


print("Total of the sector (sum of the origins in log):")
display(show(comparison[comparison["origin"] == "Total"]))
print(f"Detail by supplying bloc, {DETAIL_CASE} (blocs with VA):")
display(show(comparison[(comparison["Case"] == DETAIL_CASE) & (comparison["origin"] != "Total") & comparison["bloc"].isin(va_weights["bloc"])]))

In [17]:
comparison[(comparison["Case"] == DETAIL_CASE) & (comparison["origin"] != "Total") & comparison["bloc"].isin(va_weights["bloc"])]

,Case,bloc,origin,year,sans VA,Group,avec VA,écart
289,model4_caseB,EU27,China,2025,0.011955,Manufacturing,0.005039,-0.006917
290,model4_caseB,EU27,EU27,2025,-0.064266,Manufacturing,-0.015551,0.048715
291,model4_caseB,EU27,RestAsia,2025,0.000161,Manufacturing,0.000051,-0.000110
292,model4_caseB,EU27,RestBRICS,2025,0.000416,Manufacturing,0.000262,-0.000154
293,model4_caseB,EU27,RestOECD,2025,0.020429,Manufacturing,0.013236,-0.007193
...,...,...,...,...,...,...,...,...
4027,model4_caseB,USA,USA,2030,-0.070292,All sectors,-0.065496,0.004796
4028,model4_caseB,USA,USA,2035,-0.128083,All sectors,-0.119391,0.008692
4029,model4_caseB,USA,USA,2040,-0.185184,All sectors,-0.172682,0.012502
4030,model4_caseB,USA,USA,2045,-0.241625,All sectors,-0.225394,0.016231


In [27]:
comparison.reset_index()

,index,Case,bloc,origin,year,sans VA,Group,avec VA,écart
0,0,model2_caseA,China,Total,2025,0.077538,Manufacturing,NaN,NaN
1,1,model2_caseA,China,China,2025,0.000000,Manufacturing,NaN,NaN
2,2,model2_caseA,China,EU27,2025,0.015390,Manufacturing,NaN,NaN
3,3,model2_caseA,China,RestAsia,2025,0.000561,Manufacturing,NaN,NaN
4,4,model2_caseA,China,RestBRICS,2025,0.000251,Manufacturing,NaN,NaN
...,...,...,...,...,...,...,...,...,...
4027,4027,model4_caseB,USA,USA,2030,-0.070292,All sectors,-0.065496,0.004796
4028,4028,model4_caseB,USA,USA,2035,-0.128083,All sectors,-0.119391,0.008692
4029,4029,model4_caseB,USA,USA,2040,-0.185184,All sectors,-0.172682,0.012502
4030,4030,model4_caseB,USA,USA,2045,-0.241625,All sectors,-0.225394,0.016231


In [46]:
comparison[(comparison["origin"] == "Total") & (comparison["bloc"] == "RestOECD") & (comparison["Group"] == "All sectors") & (comparison["Case"] == "model2_caseB")]

,Case,bloc,origin,year,sans VA,Group,avec VA,écart
2580,model2_caseB,RestOECD,Total,2025,0.014308,All sectors,0.009031,-0.005277
2581,model2_caseB,RestOECD,Total,2030,0.086956,All sectors,0.054845,-0.032111
2582,model2_caseB,RestOECD,Total,2035,0.161479,All sectors,0.101774,-0.059705
2583,model2_caseB,RestOECD,Total,2040,0.237927,All sectors,0.149837,-0.088089
2584,model2_caseB,RestOECD,Total,2045,0.316364,All sectors,0.199067,-0.117297
2585,model2_caseB,RestOECD,Total,2050,0.396847,All sectors,0.249487,-0.147360


In [31]:
display(show(comparison[(comparison["Case"] == DETAIL_CASE) & (comparison["origin"] != "Total") & comparison["bloc"].isin(va_weights["bloc"])]))

2030          \
                                                       sans VA avec VA   
Case         Region (importer) Group         Origin                      
model4_caseB EU27              All sectors   China      0.0159  0.0112   
                                             EU27      -0.0474 -0.0376   
                                             RestAsia   0.0002  0.0001   
                                             RestBRICS  0.0008  0.0004   
                                             RestOECD   0.0351  0.0211   
                                             RoW        0.0011  0.0008   
                                             USA        0.0216  0.0211   
                               Manufacturing China      0.0720  0.0303   
                                             EU27      -0.3826 -0.0926   
                                             RestAsia   0.0010  0.0003   
                                             RestBRICS  0.0025  0.0016   
                                             RestOECD   0.1231  0.0797   
                                             RoW        0.0043  0.0023   
                                             USA        0.0996  0.0504   
             RestOECD          All sectors   China      0.0386  0.0176   
                                             EU27       0.0678  0.0332   
                                             RestAsia   0.0006  0.0003   
                                             RestBRICS  0.0009  0.0003   
                                             RestOECD  -0.0622 -0.0460   
                                             RoW        0.0010  0.0006   
                                             USA        0.1010  0.0521   
                               Manufacturing China      0.1223  0.0500   
                                             EU27       0.1449  0.1217   
                                             RestAsia   0.0027  0.0008   
                                             RestBRICS  0.0034  0.0011   
                                             RestOECD  -0.3500 -0.0885   
                                             RoW        0.0039  0.0013   
                                             USA        0.3184  0.1412   
             USA               All sectors   China      0.0576  0.0413   
                                             EU27       0.0687  0.0420   
                                             RestAsia   0.0006  0.0006   
                                             RestBRICS  0.0031  0.0020   
                                             RestOECD   0.1218  0.0792   
                                             RoW        0.0040  0.0034   
                                             USA       -0.0703 -0.0655   
                               Manufacturing China      0.1800  0.0490   
                                             EU27       0.1640  0.1194   
                                             RestAsia   0.0026  0.0004   
                                             RestBRICS  0.0069  0.0062   
                                             RestOECD   0.3500  0.1865   
                                             RoW        0.0131  0.0036   
                                             USA       -0.4034 -0.0918   

                                                                  2050  \
                                                         écart sans VA   
Case         Region (importer) Group         Origin                      
model4_caseB EU27              All sectors   China     -0.0047  0.0694   
                                             EU27       0.0097 -0.2008   
                                             RestAsia  -0.0000  0.0007   
                                             RestBRICS -0.0004  0.0034   
                                             RestOECD  -0.0141  0.1541   
                                             RoW       -0.0004  0.0050   
                                             USA       -0.000